In [1]:
from single_cellm.jointemb.geneformer_model import (
    GeneformerConfig,
    GeneformerModel,
    GeneformerTranscriptomeProcessor,
)
import torch
from transformers import BioGptConfig, AutoTokenizer
from single_cellm.jointemb.model import (
    TranscriptomeTextDualEncoderConfig,
    TranscriptomeTextDualEncoderModel,
)
from single_cellm.jointemb.processing import TranscriptomeTextDualEncoderProcessor
from pathlib import Path

# Initialize BioGpt and Geneformer model (with pretrained weights)
PROJECT_DIR = Path("/home/moritz/single-cellm/")
config_transcriptome = {
    "model_directory": str(PROJECT_DIR / "resources" / "geneformer-12L-30M"),
    "model_type": "geneformer"
}

config_text = {
    "model_type": "biogpt"
}

config = TranscriptomeTextDualEncoderConfig(
    projection_dim=512,
    logit_scale_init_value=2.6592,  # from the original CLIP paper. TODO does this work well for us?
    transcriptome_config=config_transcriptome,
    text_config=config_text,
)
device = torch.device("cuda")

model = TranscriptomeTextDualEncoderModel(config=config)
# manually load pretrained
model.text_model = model.text_model.from_pretrained("microsoft/biogpt")
_ = model.to(device)
# geneformer is pretrained implicitly (should change this at some point)

/home/moritz/.conda/envs/single-cellm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import anndata
import scanpy as sc

adata = anndata.read_csv(
    PROJECT_DIR / "resources/immgen/read_count_table.csv",
    first_column_names=True,
).T

In [3]:
import json
with open(PROJECT_DIR / "results/immgen/annotations.json", "r") as f:
    annot = json.load(f)

In [4]:
# Contrastive training
tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
# image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
transcriptome_processor = GeneformerTranscriptomeProcessor(nproc=4, emb_label=model.transcriptome_model.config.emb_label)
processor = TranscriptomeTextDualEncoderProcessor(transcriptome_processor, tokenizer)

In [5]:
inputs = processor(
    text=[annot[k] for k in adata.obs.index],
    transcriptomes=adata,
    return_tensors="pt",
    padding=True,
    # max_length=128
)

prepared_features has no column attribute 'filter_pass'; tokenizing all cells.


/home/moritz/Projects/single-cellm/src/single_cellm/jointemb/geneformer_model.py:51: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata_w_id.var["ensembl_id"] = annot_drop_dups.loc[
/home/moritz/Projects/single-cellm/src/single_cellm/jointemb/geneformer_model.py:129: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1695392035629/work/torch/csrc/utils/tensor_new.cpp:261.)
  tokens = torch.tensor(tokens, dtype=torch.long)


In [6]:
inputs["input_ids"]

tensor([[    2,   299, 37405,  ...,     1,     1,     1],
        [    2,   299, 37405,  ...,     1,     1,     1],
        [    2,   299, 37405,  ...,     1,     1,     1],
        ...,
        [    2,  2713,    82,  ...,     1,     1,     1],
        [    2,  2713,   116,  ...,     1,     1,     1],
        [    2,  2713,   116,  ...,     1,     1,     1]])

### There is one last small error below


In [7]:
type(model.text_model)

transformers.models.biogpt.modeling_biogpt.BioGptModel

In [9]:
# for train and inference:
outputs = model(
    input_ids=inputs.input_ids.to(device),
    attention_mask=inputs.attention_mask.to(device),
    expression_tokens=inputs.expression_tokens.to(device),
    expression_token_lengths=inputs.expression_token_lengths.to(device),
    pad_token_id = transcriptome_processor.tokenizer.gene_token_dict.get("<pad>"),
    return_loss=True,  # TODO loss computation is still failing
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42/42 [00:06<00:00,  6.01it/s]
/home/moritz/Projects/single-cellm/src/single_cellm/jointemb/model.py:246: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1695392035629/work/aten/src/ATen/native/TensorShape.cpp:3614.)
  logits_per_transcriptome = logits_per_text.T


RuntimeError: Expected floating point type for target with class probabilities, got Long

In [ ]:
# Saving the model, including its configuration
model.save_pretrained("geneformer-biogpt")

In [ ]:
# TODO needs fixed loss first (see code cell above)

loss, logits_per_transcriptome = (
    outputs.loss,
    outputs.logits_per_transcriptome,
)  # this is the transcriptome-text similarity score

# TODO training loop

# inference
outputs = model(**inputs)
logits_per_image = (
    outputs.logits_per_transcriptome
)  # this is the image-text similarity score
# probs = logits_per_image.softmax(
#     dim=1
# )  # we can take the softmax to get the label probabilities